In [ ]:
import sys,json
from pathlib import Path
cwd=Path().resolve(); repo_root=cwd.parent if cwd.name=='notebooks' else cwd
sys.path.insert(0,str(repo_root/'src'))
from dotenv import load_dotenv; load_dotenv(repo_root/'.env',override=True)

## 1. LangChain Expression Language

```python
chain = retriever | prompt | llm | parser
```

Each `|` passes the output of the left component as input to the right.
This is just Python's `__or__` operator — LangChain defines it on Runnables.

The chain reads like English: retrieve context → format prompt → call LLM → parse.

In [ ]:
from rag.retrieval.vector_retriever import VectorRetriever
from rag.chains.rag_chain import build_rag_chain, format_docs
from rag.config.prompts import RAG_PROMPT
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from rag.config.settings import get_settings
settings  = get_settings()
retriever = VectorRetriever()
llm = ChatOpenAI(model=settings.openai_model, temperature=0, streaming=True,
                 openai_api_key=settings.openai_api_key.get_secret_value())
chain = (
    {'context': retriever.as_langchain_retriever() | format_docs,
     'question': RunnablePassthrough()}
    | RAG_PROMPT | llm | StrOutputParser()
)
print('✅ Chain built:', type(chain).__name__)

In [ ]:
question = 'What is the total fertility rate in Nigeria according to the 2021 DHS?'
print(f'Q: {question}')
print()
print('A: ', end='')
answer = chain.invoke({'question': question})
print(answer)

In [ ]:
print('Streaming answer:')
for token in chain.stream({'question': question}):
    print(token, end='', flush=True)
print()

In [ ]:
docs = retriever.retrieve(question)
print(f'Retrieved {len(docs)} chunks')
for i,d in enumerate(docs[:3],1):
    print(f'\n[Source {i}] score={d.metadata["similarity_score"]} | {d.metadata["country"]} {d.metadata["year"]} p{d.metadata["page_number"]}')
    print(d.page_content[:300])

In [ ]:
# Ask something NOT in our corpus — see how the system responds
out_of_scope = 'What is the GDP per capita of Mars?'
print(f'Q: {out_of_scope}')
print()
print('A: ', end='')
for token in chain.stream({'question': out_of_scope}):
    print(token, end='', flush=True)
print()
print()
print('The system prompt instructs the LLM to decline out-of-scope questions.')
print('Without it, the LLM would answer from training data = hallucination.')

## ✅ Episode 5 complete

**Episode 6:** Hybrid search — combining BM25 keyword search with vector similarity.
We'll show why 'DHS 2022 Kenya' retrieves better with BM25.